## PLN 2026 - Trabalho da disciplina
Baseado no exemplo da Aula 7 (Prof. Maiko Spiess)

---

## Representações densas e busca semântica em notícias do Lance!

Este notebook aplica à base de notícias do nosso projeto ([github.com/pkalbuquerque/pln](https://github.com/pkalbuquerque/pln)) o mesmo roteiro da Aula 7, que usou a base sintética de animais. Construímos duas representações densas e um buscador por similaridade:

1. **Word2Vec** (Skip-gram), treinado com o próprio corpus de notícias;
2. **BERTimbau** (`neuralmind/bert-base-portuguese-cased`) com *mean pooling*;
3. **TF-IDF** como linha de base (*baseline*).

Para uma mesma consulta livre, o sistema devolve as 3 notícias mais próximas em cada espaço vetorial, e podemos comparar os rankings. O mecanismo de busca (similaridade do cosseno) é o mesmo nos três casos, e só a **representação** muda.

Além das consultas de exemplo, fazemos uma avaliação quantitativa que a base de animais não permitia. Como cada notícia tem uma `secao` (clube ou editoria), medimos com que frequência os vizinhos mais próximos de uma notícia pertencem à mesma seção.

In [1]:
# No Colab, pandas, scikit-learn, numpy, nltk e torch já vêm instalados.
# Localmente: pip install gensim transformers torch scikit-learn tabulate
%pip -q install gensim transformers tabulate

# Importar o torch antes do pandas evita erro de DLL em alguns Windows
import torch


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Carregando a base de dados

Usamos `outputs/noticias_lance_processadas.jsonl`, gerado por `python pipeline.py`. Esse arquivo reúne tudo o que precisamos em um só lugar:

- `texto_original`: texto com maiúsculas e pontuação, que será usado no BERTimbau;
- `tokens_sem_stopwords`: tokens já limpos pelo `preprocessar.py`, que serão usados no Word2Vec;
- `titulo`, `secao`, `id`, `url`.

Assim como na aula, o **Word2Vec** recebe os tokens normalizados e sem *stopwords*, e o **BERT** recebe o texto original.

In [2]:
import json
import os
import pandas as pd

# No Colab, pede o upload do arquivo; localmente, lê direto de outputs/.
try:
    from google.colab import files

    print("Selecione o arquivo outputs/noticias_lance_processadas.jsonl")
    uploaded = files.upload()
    nome_arquivo = next(iter(uploaded))

except ImportError:
    nome_arquivo = "outputs/noticias_lance_processadas.jsonl"
    print(f"Ambiente local detectado. Lendo diretamente: {nome_arquivo}")

with open(nome_arquivo, encoding="utf-8") as f:
    df = pd.DataFrame([json.loads(linha) for linha in f if linha.strip()])

df["secao"] = df["secao"].fillna("Sem seção")
df = df.reset_index(drop=True)

print(f"Quantidade de notícias: {len(df)}")
df[["id", "titulo", "secao", "quantidade_tokens_limpos"]].head()

Ambiente local detectado. Lendo diretamente: outputs/noticias_lance_processadas.jsonl
Quantidade de notícias: 89


,id,titulo,secao,quantidade_tokens_limpos
0,4113d327c94a3a03,Palmeiras: Conmebol define gancho de Abel Ferr...,Palmeiras,102
1,fc1182644d3d11ff,Reinaldo pede chance na Seleção e cita Rodinei...,Futebol Nacional,237
2,d4849db3da5014b5,Arrascaeta atualiza torcida do Flamengo após c...,Flamengo,196
3,8d2da6471942ac6c,Série A: clube rescinde com atacante após disc...,Futebol Nacional,172
4,f6e30ca77e1b3f0d,"Torcedor do Palmeiras, Lito Sousa morre aos 59...",Fora de Campo,215


In [3]:
PASTA_TABELAS = "tabelas_exportadas"
os.makedirs(PASTA_TABELAS, exist_ok=True)


def baixar_tabela(tabela, nome_arquivo, index=False):
    """Salva um DataFrame em CSV e, no Colab, dispara o download."""
    caminho = os.path.join(PASTA_TABELAS, nome_arquivo)
    tabela.to_csv(caminho, index=index, encoding="utf-8")
    print(f"Tabela salva em: {caminho}")
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        pass

## Texto de cada notícia

Cada documento é formado pelo **título** seguido do **corpo** da matéria.

- Para o Word2Vec usamos `tokens_sem_stopwords` (corpo já processado) mais os tokens do título, passados pelas mesmas funções de tokenização, normalização e remoção de *stopwords* da aula.
- Para o BERTimbau usamos o texto original `titulo + corpo`. O modelo aceita no máximo 512 *subwords*, e truncamos em 256 para manter a execução rápida em CPU. As notícias têm em média cerca de 200 palavras, então a maior parte do conteúdo é preservada.

In [4]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stopwords_pt = set(stopwords.words("portuguese"))


def tokenizar(texto):
    return texto.split()


def normalizar(tokens):
    resultado = []
    for token in tokens:
        token = token.lower()
        token = re.sub(r"[^\wÀ-ÿ-]", "", token)
        if token:
            resultado.append(token)
    return resultado


def remover_stopwords(tokens):
    return [t for t in tokens if t not in stopwords_pt]


def preparar_consulta(texto):
    """Mesma preparação aplicada aos documentos do Word2Vec."""
    return remover_stopwords(normalizar(tokenizar(texto)))


# Documentos para o Word2Vec: tokens do título + tokens do corpo
df["tokens_w2v"] = [
    preparar_consulta(titulo) + list(corpo)
    for titulo, corpo in zip(df["titulo"], df["tokens_sem_stopwords"])
]

# Documentos para o BERT: texto original (título + corpo)
df["texto_bert"] = df["titulo"] + ". " + df["texto_original"]

df[["titulo", "tokens_w2v"]].head()

,titulo,tokens_w2v
0,Palmeiras: Conmebol define gancho de Abel Ferr...,"[palmeiras, conmebol, define, gancho, abel, fe..."
1,Reinaldo pede chance na Seleção e cita Rodinei...,"[reinaldo, pede, chance, seleção, cita, rodine..."
2,Arrascaeta atualiza torcida do Flamengo após c...,"[arrascaeta, atualiza, torcida, flamengo, após..."
3,Série A: clube rescinde com atacante após disc...,"[série, clube, rescinde, atacante, após, discu..."
4,"Torcedor do Palmeiras, Lito Sousa morre aos 59...","[torcedor, palmeiras, lito, sousa, morre, 59, ..."


## Representação 1 — Word2Vec

Treinamos um **Skip-gram** com o corpus de notícias. A base tem poucas dezenas de textos, então o modelo tem pouco material para aprender relações semânticas. Os resultados devem ser lidos como demonstração do mecanismo, e não como um modelo de alta qualidade.

A diferença em relação à base de animais é que o vocabulário aqui é cheio de **nomes próprios** (jogadores, técnicos, clubes) que aparecem poucas vezes, o que torna o aprendizado ainda mais difícil.

In [5]:
from gensim.models import Word2Vec

modelo_w2v = Word2Vec(
    sentences=df["tokens_w2v"].tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,          # Skip-gram
    seed=42,
    workers=1,     # reprodutibilidade
    epochs=100
)

print("Tamanho do vocabulário:", len(modelo_w2v.wv))
print("Dimensões dos vetores:", modelo_w2v.vector_size)

# Palavras mais próximas de alguns termos, para inspecionar o que o modelo aprendeu
for palavra in ["flamengo", "gol", "técnico"]:
    if palavra in modelo_w2v.wv:
        vizinhas = modelo_w2v.wv.most_similar(palavra, topn=5)
        print(f"\n{palavra}:", [(p, round(s, 2)) for p, s in vizinhas])

Tamanho do vocabulário: 5079
Dimensões dos vetores: 100

flamengo: [('regatas', 0.44), ('stf', 0.44), ('fux', 0.44), ('sobre', 0.43), ('esclarece', 0.42)]

gol: [('segundos', 0.7), ('oitava', 0.68), ('tomamos', 0.68), ('bolivianos', 0.68), ('parada', 0.67)]

técnico: [('auxiliares', 0.65), ('futuros', 0.62), ('rivais', 0.6), ('franclim', 0.59), ('carpini', 0.59)]


### Do vetor da palavra ao vetor do documento

Como na aula, o vetor do documento é a **média** dos vetores das suas palavras:

$$
v_{doc} = \frac{1}{n}\sum_{i=1}^{n}v_i
$$

In [6]:
import numpy as np


def vetor_documento_word2vec(tokens):
    vetores = [modelo_w2v.wv[t] for t in tokens if t in modelo_w2v.wv]
    if not vetores:
        return np.zeros(modelo_w2v.vector_size, dtype=np.float32)
    return np.mean(vetores, axis=0)


embeddings_word2vec = np.vstack(df["tokens_w2v"].apply(vetor_documento_word2vec))
print("Formato da matriz Word2Vec:", embeddings_word2vec.shape)

Formato da matriz Word2Vec: (89, 100)


## Representação 2 — BERTimbau

Usamos o **BERTimbau Base** com *mean pooling* sobre os estados do último nível. O modelo é *cased*, então a capitalização conta e por isso passamos o texto original, sem converter para minúsculas.

O primeiro uso baixa cerca de 400 MB do Hugging Face.

In [7]:
import torch
from transformers import AutoTokenizer, AutoModel

MODELO_BERT = "neuralmind/bert-base-portuguese-cased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

tokenizer_bert = AutoTokenizer.from_pretrained(MODELO_BERT, do_lower_case=False)
modelo_bert = AutoModel.from_pretrained(MODELO_BERT).to(device)
modelo_bert.eval()
print("Modelo carregado:", MODELO_BERT)

Dispositivo: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado: neuralmind/bert-base-portuguese-cased


In [8]:
def mean_pooling(last_hidden_state, attention_mask):
    mascara = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    soma = torch.sum(last_hidden_state * mascara, dim=1)
    quantidade = torch.clamp(mascara.sum(dim=1), min=1e-9)
    return soma / quantidade


def gerar_embeddings_bert(textos, batch_size=8, max_length=256):
    todos = []
    for inicio in range(0, len(textos), batch_size):
        lote = textos[inicio:inicio + batch_size]
        entradas = tokenizer_bert(
            lote, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        )
        entradas = {k: v.to(device) for k, v in entradas.items()}
        with torch.no_grad():
            saida = modelo_bert(**entradas)
        todos.append(mean_pooling(saida.last_hidden_state, entradas["attention_mask"]).cpu().numpy())
    return np.vstack(todos)


embeddings_bert = gerar_embeddings_bert(df["texto_bert"].tolist())
print("Formato da matriz BERTimbau:", embeddings_bert.shape)

Formato da matriz BERTimbau: (89, 768)


## Baseline — TF-IDF

Mantemos o TF-IDF (que vimos na Aula 6) como referência, construído sobre os mesmos tokens usados no Word2Vec.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["tokens_w2v"])
print("Formato da matriz TF-IDF:", tfidf_matrix.shape)

resumo_representacoes = pd.DataFrame({
    "representacao": ["TF-IDF", "Word2Vec", "BERTimbau"],
    "dimensoes": [tfidf_matrix.shape[1], embeddings_word2vec.shape[1], embeddings_bert.shape[1]],
    "tipo": ["esparsa, lexical", "densa, estática", "densa, contextual"],
})
baixar_tabela(resumo_representacoes, "01_representacoes.csv")
resumo_representacoes

Formato da matriz TF-IDF: (89, 5079)
Tabela salva em: tabelas_exportadas\01_representacoes.csv


,representacao,dimensoes,tipo
0,TF-IDF,5079,"esparsa, lexical"
1,Word2Vec,100,"densa, estática"
2,BERTimbau,768,"densa, contextual"


## Busca por similaridade

O procedimento é igual para as três representações: representar a consulta, calcular a similaridade do cosseno com todas as notícias, ordenar e ficar com as `k` primeiras.

Como na aula, as pontuações só servem para ordenar **dentro** de cada representação, e não é correto dizer que 0,80 no BERT é "melhor" que 0,40 no TF-IDF.

In [10]:
def buscar_top_k(vetor_consulta, matriz_documentos, k=3):
    similaridades = cosine_similarity(
        np.asarray(vetor_consulta).reshape(1, -1), matriz_documentos
    )[0]
    indices = np.argsort(similaridades)[::-1][:k]

    resultado = df.iloc[indices][["id", "titulo", "secao"]].copy()
    resultado.insert(0, "posicao", range(1, len(resultado) + 1))
    resultado["similaridade"] = similaridades[indices].round(3)
    return resultado


def representar_consulta_word2vec(texto):
    tokens = preparar_consulta(texto)
    conhecidos = [t for t in tokens if t in modelo_w2v.wv]
    oov = [t for t in tokens if t not in modelo_w2v.wv]
    if not conhecidos:
        return None, conhecidos, oov
    return np.mean([modelo_w2v.wv[t] for t in conhecidos], axis=0), conhecidos, oov


def representar_consulta_bert(texto):
    return gerar_embeddings_bert([texto])[0]


def representar_consulta_tfidf(texto):
    return tfidf_vectorizer.transform([preparar_consulta(texto)])


def buscar_comparando(consulta, k=3, mostrar=True):
    resultados = {}

    vetor_w2v, conhecidos, oov = representar_consulta_word2vec(consulta)
    if vetor_w2v is not None:
        resultados["Word2Vec"] = buscar_top_k(vetor_w2v, embeddings_word2vec, k)
    resultados["BERTimbau"] = buscar_top_k(representar_consulta_bert(consulta), embeddings_bert, k)
    resultados["TF-IDF"] = buscar_top_k(representar_consulta_tfidf(consulta).toarray(), tfidf_matrix.toarray(), k)

    if mostrar:
        print(f"\n{'=' * 60}\nConsulta: {consulta}\n{'=' * 60}")
        print(f"Word2Vec - tokens conhecidos: {conhecidos or 'nenhum'} | OOV: {oov or 'nenhum'}")
        for nome, tabela in resultados.items():
            print(f"\n--- {nome} ---")
            print(tabela.to_markdown(index=False))
    return resultados

## Executando buscas

Usamos uma lista fixa de consultas para que o notebook seja reprodutível. As duas primeiras repetem o vocabulário das notícias, e as demais usam palavras diferentes das que aparecem no texto, que é onde a busca semântica deveria ajudar. Na última célula há um campo para digitar uma consulta livre.

In [11]:
consultas = [
    "notícias sobre o Flamengo",
    "técnico punido pela Conmebol",
    "jogador machucado fora dos gramados",
    "time está perto de contratar um reforço",
    "clube precisa vencer para escapar do rebaixamento",
]

registro_buscas = []
for consulta in consultas:
    resultados = buscar_comparando(consulta, k=3)
    for representacao, tabela in resultados.items():
        t = tabela.copy()
        t.insert(0, "representacao", representacao)
        t.insert(0, "consulta", consulta)
        registro_buscas.append(t)

df_buscas = pd.concat(registro_buscas, ignore_index=True)
baixar_tabela(df_buscas, "02_resultados_buscas.csv")


Consulta: notícias sobre o Flamengo
Word2Vec - tokens conhecidos: ['notícias', 'sobre', 'flamengo'] | OOV: nenhum

--- Word2Vec ---
|   posicao | id               | titulo                                                          | secao    |   similaridade |
|----------:|:-----------------|:----------------------------------------------------------------|:---------|---------------:|
|         1 | 188885ed8b53d361 | Flamengo se manifesta sobre mudança de jogo contra o Santos     | Flamengo |          0.735 |
|         2 | 7ba51192abd9ceaf | Flamengo volta a treinar nesta quarta-feira; veja a programação | Flamengo |          0.711 |
|         3 | ebc7d078ef0127f7 | Flamengo é aceito por Fux em processos do STF sobre bets        | Flamengo |          0.71  |

--- BERTimbau ---
|   posicao | id               | titulo                                                    | secao    |   similaridade |
|----------:|:-----------------|:----------------------------------------------------------|


Consulta: jogador machucado fora dos gramados
Word2Vec - tokens conhecidos: ['jogador', 'gramados'] | OOV: ['machucado']

--- Word2Vec ---
|   posicao | id               | titulo                                                         | secao    |   similaridade |
|----------:|:-----------------|:---------------------------------------------------------------|:---------|---------------:|
|         1 | 42a2ede4e9fe1ee9 | Vasco se aproxima de esvaziar Departamento Médico              | Vasco    |          0.579 |
|         2 | a11db36fc11e1e06 | Cruzeiro: Pec tem projeção de retorno aos treinos em campo     | Cruzeiro |          0.578 |
|         3 | 7a2427d03f65a2c9 | Vasco aposta em 'reforço caseiro' para reta final da temporada | Vasco    |          0.564 |

--- BERTimbau ---
|   posicao | id               | titulo                                                    | secao      |   similaridade |
|----------:|:-----------------|:-------------------------------------------------------


Consulta: clube precisa vencer para escapar do rebaixamento
Word2Vec - tokens conhecidos: ['clube', 'precisa', 'vencer', 'rebaixamento'] | OOV: ['escapar']

--- Word2Vec ---
|   posicao | id               | titulo                                                      | secao    |   similaridade |
|----------:|:-----------------|:------------------------------------------------------------|:---------|---------------:|
|         1 | ccc11bad315bca31 | Artur Jorge exalta jogador do Cruzeiro: 'Estará na Seleção' | Cruzeiro |          0.72  |
|         2 | 3d6d757ca0481329 | Artur Jorge exalta Matheus Pereira: "Melhor do Brasil"      | Cruzeiro |          0.716 |
|         3 | e8a47619f138b8e2 | Artur Jorge avalia desafio do Cruzeiro para alcançar rivais | Cruzeiro |          0.7   |

--- BERTimbau ---
|   posicao | id               | titulo                                                                                                   | secao     |   similaridade |
|----------:|:--------

## Avaliação quantitativa: os vizinhos são da mesma seção?

A base de animais não permitia medir qualidade, mas aqui cada notícia tem uma `secao`. Para cada notícia usamos o próprio texto como consulta, pegamos as 3 mais parecidas (excluindo ela mesma) e calculamos a fração delas que pertence à **mesma seção**.

Essa métrica é só uma aproximação: duas notícias de seções diferentes podem tratar do mesmo assunto, como um clássico entre dois clubes. Mas ela dá um número comparável entre as representações.

In [12]:
def precisao_vizinhos(matriz, k=3):
    sim = cosine_similarity(matriz)
    np.fill_diagonal(sim, -np.inf)  # exclui a própria notícia
    vizinhos = np.argsort(sim, axis=1)[:, ::-1][:, :k]
    secoes = df["secao"].to_numpy()
    acertos = (secoes[vizinhos] == secoes[:, None])
    return acertos.mean(), acertos.mean(axis=1)


avaliacao = []
por_documento = {}
for nome, matriz in {
    "TF-IDF": tfidf_matrix.toarray(),
    "Word2Vec": embeddings_word2vec,
    "BERTimbau": embeddings_bert,
}.items():
    media, por_doc = precisao_vizinhos(matriz)
    por_documento[nome] = por_doc
    avaliacao.append({"representacao": nome, "precisao_mesma_secao_top3": round(media, 3)})

# Referência: o que se esperaria ao acaso
proporcoes = df["secao"].value_counts(normalize=True)
acaso = float((proporcoes ** 2).sum())
avaliacao.append({"representacao": "acaso (referência)", "precisao_mesma_secao_top3": round(acaso, 3)})

df_avaliacao = pd.DataFrame(avaliacao)
baixar_tabela(df_avaliacao, "03_precisao_vizinhos.csv")
df_avaliacao

Tabela salva em: tabelas_exportadas\03_precisao_vizinhos.csv

,representacao,precisao_mesma_secao_top3
0,TF-IDF,0.521
1,Word2Vec,0.390
2,BERTimbau,0.228
3,acaso (referência),0.101


In [13]:
# Desempenho por seção (média dos documentos de cada seção)
tabela_secao = pd.DataFrame(por_documento)
tabela_secao["secao"] = df["secao"]
tabela_secao = tabela_secao.groupby("secao").mean().round(2)
tabela_secao["n_noticias"] = df["secao"].value_counts()
tabela_secao = tabela_secao.sort_values("n_noticias", ascending=False)

baixar_tabela(tabela_secao.reset_index(), "04_precisao_por_secao.csv")
tabela_secao

Tabela salva em: tabelas_exportadas\04_precisao_por_secao.csv


,TF-IDF,Word2Vec,BERTimbau,n_noticias
secao,,,,
Futebol Nacional,0.36,0.29,0.29,15
Flamengo,0.74,0.69,0.33,13
Corinthians,0.67,0.42,0.21,8
Cruzeiro,0.71,0.67,0.46,8
Vasco,0.88,0.54,0.25,8
Santos,0.52,0.29,0.10,7
Atlético Mineiro,0.56,0.28,0.17,6
Fluminense,0.44,0.28,0.11,6
São Paulo,0.39,0.28,0.17,6


## Concordância entre as representações

Para cada consulta de exemplo, quantas das 3 notícias recuperadas são as mesmas em duas representações?

In [14]:
linhas = []
for consulta in consultas:
    res = buscar_comparando(consulta, k=3, mostrar=False)
    conjuntos = {nome: set(t["id"]) for nome, t in res.items()}
    linhas.append({
        "consulta": consulta,
        "TFIDF_x_BERT": len(conjuntos["TF-IDF"] & conjuntos["BERTimbau"]),
        "TFIDF_x_W2V": len(conjuntos["TF-IDF"] & conjuntos.get("Word2Vec", set())),
        "W2V_x_BERT": len(conjuntos.get("Word2Vec", set()) & conjuntos["BERTimbau"]),
    })

df_concordancia = pd.DataFrame(linhas)
baixar_tabela(df_concordancia, "05_concordancia_buscas.csv")
df_concordancia

Tabela salva em: tabelas_exportadas\05_concordancia_buscas.csv


,consulta,TFIDF_x_BERT,TFIDF_x_W2V,W2V_x_BERT
0,notícias sobre o Flamengo,0,3,0
1,técnico punido pela Conmebol,2,3,2
2,jogador machucado fora dos gramados,0,2,0
3,time está perto de contratar um reforço,1,1,1
4,clube precisa vencer para escapar do rebaixamento,0,2,0


## Sua própria consulta

Rode a célula abaixo e digite uma consulta livre. No Colab ela abre uma caixa de texto; para pular, deixe vazia.

In [15]:
try:
    consulta_livre = input("Digite sua consulta (ou deixe vazio para pular): ").strip()
except Exception:  # execução sem interface (ex.: nbconvert)
    consulta_livre = ""
if consulta_livre:
    buscar_comparando(consulta_livre, k=3)

## Conclusão

Aplicamos à base de notícias do Lance! o roteiro da Aula 7: **Word2Vec** (treinado no próprio corpus), **BERTimbau + mean pooling** e um **baseline TF-IDF**, todos com o mesmo mecanismo de busca por similaridade do cosseno. A base tem 89 notícias coletadas em 01/10/2026.

### O que observamos

- **Avaliação por seção** (vizinhos top-3 da mesma seção; acaso ≈ 0,10): TF-IDF **0,52**, Word2Vec **0,39**, BERTimbau **0,23**. Essa métrica favorece quem casa o nome do clube, e é onde o TF-IDF é forte. O resultado mostra que, para agrupar por clube, o casamento lexical continua competitivo, e não que o BERT seja pior em geral.
- **Consultas que repetem o vocabulário** ("notícias sobre o Flamengo"): as três representações acertam a seção. Os rankings são diferentes: o BERT devolve três notícias de Flamengo que o TF-IDF não trouxe (concordância 0 de 3).
- **Consultas semânticas** ("jogador machucado fora dos gramados"): o TF-IDF e o Word2Vec recuperam notícias do Vasco ligadas a "Departamento Médico". O BERT recupera notícias de clubes **diferentes** (Santos, Flamengo, Fluminense) que tratam de lesão e "DM". Ele capta o **tema** e ignora o clube, o que é bom para busca por assunto e ruim para a métrica por seção.
- **Palavras fora do vocabulário**: no Word2Vec, "machucado", "contratar" e "escapar" são OOV, porque o modelo só conhece o que viu em 89 textos. O BERT usa subpalavras e não sofre disso. Na consulta de rebaixamento, o Word2Vec e o TF-IDF devolveram notícias do Cruzeiro sem relação com rebaixamento (só casaram palavras como "jogador" e "clube"), e o BERT trouxe notícias sobre campanha e classificação no Brasileirão, embora também sem uma resposta direta.
- **Word2Vec com poucos dados**: os vizinhos de "flamengo" (regatas, stf, fux) refletem as notícias da coleta, e não uma noção geral do idioma.

### Respostas às questões da aula

1. *Onde coincidem?* Em consultas que citam termos que existem literalmente nas notícias (clube, Conmebol).
2. *Onde divergem?* Nas consultas por tema, sem palavras compartilhadas com o texto.
3. *Efeito do corpus pequeno?* Word2Vec fica instável e dominado por nomes próprios raros.
4. *O BERTimbau geral é consistente após o mean pooling?* Em parte: capta temas, mas as similaridades ficam todas em faixa alta e próxima (0,45–0,63), o que indica pouca separação. Um modelo ajustado para *sentence embeddings* provavelmente se sairia melhor.
5. *Quando o BERT recupera o que o Word2Vec não consegue?* Quando a consulta usa palavras ausentes do corpus (OOV) ou sinônimos.
6. *Corpus maior?* O Word2Vec melhoraria bastante; o BERT já vem pré-treinado.
7. *Índice aproximado?* Com 89 documentos a busca exaustiva é instantânea. Com milhões, seria preciso um índice como HNSW.

### Limitações

A avaliação é pequena (89 notícias, 12 seções, algumas com 2 a 5 exemplos) e o corte em 256 *subwords* descarta o final das notícias mais longas. Os números mudam a cada coleta, pois o sitemap do Lance! é renovado diariamente.
